In [2]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import itertools

In [3]:
num_class = [47, 10, 43, 10, 45, 196, 397, 10]
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"]
domain = "Base_Fine_Tuned"
transform_type = "Standard" # Base_Fine_Tuned_Classifier | Standard
model_name = "CLIP_ViT_Vision"
refining_type = "Standard" # Increment_Training | Standard
refiner = "Fine_Tuned" # Fine_Tuned | Reverse_Probe
indices = [i for i in range(12)]

In [4]:
# Store all combinations of lengths 2 to 8
all_combinations = {}

for r in range(1, 9):  # lengths 2 through 8
    combos = list(itertools.combinations(dataset_name, r))
    all_combinations[r] = combos

In [ ]:
task_matrix = {}

for num_sets in range(1,9):
    task_matrix[num_sets] = {}
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        W_folder = f"../Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Task_Matrices/{num_sets}/{domain}/{matrix_name}"
        task_matrix[num_sets][matrix_name] = {}
        for rep in range(1,6):
            task_matrix[num_sets][matrix_name][rep] = {"U": {}, "S": {}, "Vh": {}}
            W = np.load(f"{W_folder}/{rep}.npy")
            for i in indices:
                task_matrix[num_sets][matrix_name][rep]["U"][i], task_matrix[num_sets][matrix_name][rep]["S"][i], task_matrix[num_sets][matrix_name]["Vh"][i] = np.linalg.svd(W[i])

In [ ]:
import scipy
from scipy.stats import sem

energy_capture = {}

for num_sets in range(1,9):
    energy_capture[num_sets] = {}
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        k_avg = []
        for rep in range(1,6):
            energy = task_matrix[num_sets][combo][rep]["S"][11] ** 2
            cumulative = np.cumsum(energy)
            total = cumulative[-1]
            frac = cumulative / total

            threshold = 0.95
            k = np.searchsorted(frac, threshold) + 1
            print("k: {:0%} energy:".format(threshold), k)
            k_avg.append(k)
        k_mean = np.mean(k_avg)
        ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(k_avg)-1, loc=k_mean, scale=sem(k_avg))
        print(f"{matrix_name} - Mean: {k_mean} +- {ci_high-k_mean}")
        energy_capture[num_sets][matrix_name] = (k_mean, ci_low, ci_high)

In [ ]:
import json

with open("k_95%_Task_Matrices.json", "w") as f:
    json.dump(energy_capture, f, indent=2)

In [24]:
energy_capture = {}

for num_sets in range(1,9):
    energy_capture[num_sets] = {}
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        energy = task_matrix[num_sets][matrix_name]["S"][11] ** 2
        cumulative = np.cumsum(energy)
        total = cumulative[-1]
        frac = cumulative / total
        threshold = 0.95

        k = np.searchsorted(frac, threshold) + 1
        print(f"{matrix_name} k: {threshold} energy: {k}")
        energy_capture[num_sets][matrix_name] = k

DTD_ k: 0.95 energy: 316
EuroSAT_ k: 0.95 energy: 91
GTSRB_ k: 0.95 energy: 48
MNIST_ k: 0.95 energy: 25
RESISC45_ k: 0.95 energy: 255
Stanford_Cars_ k: 0.95 energy: 333
SUN397_ k: 0.95 energy: 545
SVHN_ k: 0.95 energy: 18
DTD_EuroSAT_ k: 0.95 energy: 142
DTD_GTSRB_ k: 0.95 energy: 71
DTD_MNIST_ k: 0.95 energy: 26
DTD_RESISC45_ k: 0.95 energy: 305
DTD_Stanford_Cars_ k: 0.95 energy: 398
DTD_SUN397_ k: 0.95 energy: 542
DTD_SVHN_ k: 0.95 energy: 38
EuroSAT_GTSRB_ k: 0.95 energy: 66
EuroSAT_MNIST_ k: 0.95 energy: 32
EuroSAT_RESISC45_ k: 0.95 energy: 147
EuroSAT_Stanford_Cars_ k: 0.95 energy: 162
EuroSAT_SUN397_ k: 0.95 energy: 373
EuroSAT_SVHN_ k: 0.95 energy: 38
GTSRB_MNIST_ k: 0.95 energy: 43
GTSRB_RESISC45_ k: 0.95 energy: 96
GTSRB_Stanford_Cars_ k: 0.95 energy: 75
GTSRB_SUN397_ k: 0.95 energy: 226
GTSRB_SVHN_ k: 0.95 energy: 46
MNIST_RESISC45_ k: 0.95 energy: 46
MNIST_Stanford_Cars_ k: 0.95 energy: 21
MNIST_SUN397_ k: 0.95 energy: 73
MNIST_SVHN_ k: 0.95 energy: 20
RESISC45_Stanford_Car

In [ ]:
U = {}
S = {}
Vh = {}

for i in indices:
    U[i] = []
    S[i] = []
    Vh[i] = []
    for j in task_matrixes:
        arr = np.array(j["W"][i])
        u, s, vh = np.linalg.svd(arr)
        U[i].append(np.array(u))
        S[i].append(np.array(s))
        Vh[i].append(np.array(vh))
        

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(S[11][6], marker='.', linewidth=1)
plt.title('Singular values (linear scale)')
plt.xlabel('index i')
plt.ylabel(r'$\sigma_i$')
plt.grid(True)

plt.figure(figsize=(8,4))
plt.semilogy(S[11][6], marker='.', linewidth=1)
plt.title('Singular values (log scale)')
plt.xlabel('index i')
plt.ylabel(r'$\sigma_i$ (log scale)')
plt.grid(True)
plt.show()

In [21]:
norm = np.linalg.norm(task_matrix[1]["MNIST_"]["S"][11])
print(np.sqrt(norm))

16.096107


In [ ]:
energy = task_matrix[1]["MNIST_"]["S"][11] ** 2
cumulative = np.cumsum(energy)
total = cumulative[-1]
frac = cumulative / total

threshold = 0.95
k = np.searchsorted(frac, threshold) + 1
print("k: {:0%} energy:".format(threshold), k)

k: 95.000000% energy: 25


In [ ]:
energy = S[11][6]**2
cumulative = np.cumsum(energy)
total = cumulative[-1]
frac = cumulative / total  # fraction of Frobenius energy captured

# Pick k for a threshold
threshold = 0.95
k = np.searchsorted(frac, threshold) + 1  # +1 because searchsorted returns idx
print("k for {:.0%} energy:".format(threshold), k)

# Plot fraction
plt.figure(figsize=(8,4))
plt.plot(frac, linewidth=2)
plt.axhline(threshold, color='red', linestyle='--')
plt.xlabel('k')
plt.ylabel('Fraction of energy captured')
plt.grid(True)
plt.show()

In [ ]:
# Operator Norm
print(np.linalg.norm(S[11][6], ord=2))